In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        fname=os.path.join(dirname, filename)
        print(fname)
        if "sub" in filename:
            df_sub=pd.read_csv(fname)
        elif "train" in filename:
            df_train=pd.read_csv(fname)
        elif "test" in filename:
            df_test=pd.read_csv(fname)
        else:
            df_origin=pd.read_csv(fname)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e3/sample_submission.csv
/kaggle/input/playground-series-s5e3/train.csv
/kaggle/input/playground-series-s5e3/test.csv
/kaggle/input/rainfall-prediction-using-machine-learning/Rainfall.csv


In [2]:
# !pip install --upgrade scikit-learn==1.4 --no-cache-dir
# !pip install cmaes


In [3]:
import pandas as pd
import numpy as np
from functools import partial
from copy import deepcopy
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# Import libraries for gradient boosting
import optuna
from optuna.samplers import CmaEsSampler
from sklearn.base import BaseEstimator, TransformerMixin
import xgboost as xgb
import lightgbm as lgb

from catboost import CatBoost, CatBoostRegressor, CatBoostRegressor
from catboost import Pool
from category_encoders import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import log_loss, auc,roc_auc_score #root_mean_squared_error,
from sklearn.preprocessing import StandardScaler#,TargetEncoder
from sklearn.model_selection import StratifiedKFold, KFold


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from transformers import AdamW
from sklearn.model_selection import KFold, GroupKFold
from tqdm.auto import tqdm

import optuna
from optuna.trial import TrialState

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")


In [4]:
df_origin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   day                     366 non-null    int64  
 1   pressure                366 non-null    float64
 2   maxtemp                 366 non-null    float64
 3   temparature             366 non-null    float64
 4   mintemp                 366 non-null    float64
 5   dewpoint                366 non-null    float64
 6   humidity                366 non-null    int64  
 7   cloud                   366 non-null    int64  
 8   rainfall                366 non-null    object 
 9   sunshine                366 non-null    float64
 10           winddirection  365 non-null    float64
 11  windspeed               365 non-null    float64
dtypes: float64(8), int64(3), object(1)
memory usage: 34.4+ KB


In [5]:
df_origin.describe()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
count,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,365.000000,365.000000
mean,15.756831,1013.742623,26.191257,23.747268,21.894536,19.989071,80.177596,71.128415,4.419399,101.506849,21.536986
std,8.823592,6.414776,5.978343,5.632813,5.594153,5.997021,10.062470,21.798012,3.934398,81.723724,10.069712
min,1.000000,998.500000,7.100000,4.900000,3.100000,-0.400000,36.000000,0.000000,0.000000,10.000000,4.400000
25%,8.000000,1008.500000,21.200000,18.825000,17.125000,16.125000,75.000000,58.000000,0.500000,40.000000,13.700000
50%,16.000000,1013.000000,27.750000,25.450000,23.700000,21.950000,80.500000,80.000000,3.500000,70.000000,20.500000
75%,23.000000,1018.100000,31.200000,28.600000,26.575000,25.000000,87.000000,88.000000,8.200000,190.000000,27.900000
max,31.000000,1034.600000,36.300000,32.400000,30.000000,26.700000,98.000000,100.000000,12.100000,350.000000,59.500000


In [6]:
origin_std=df_origin.describe().iloc[2,]
origin_std

day                        8.823592
pressure                   6.414776
maxtemp                    5.978343
temparature                5.632813
mintemp                    5.594153
dewpoint                   5.997021
humidity                  10.062470
cloud                     21.798012
sunshine                   3.934398
         winddirection    81.723724
windspeed                 10.069712
Name: std, dtype: float64

In [7]:
index=origin_std.keys()
index = [ a.strip() for a in index ]
origin_std.index=index

In [8]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2190 entries, 0 to 2189
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             2190 non-null   int64  
 1   day            2190 non-null   int64  
 2   pressure       2190 non-null   float64
 3   maxtemp        2190 non-null   float64
 4   temparature    2190 non-null   float64
 5   mintemp        2190 non-null   float64
 6   dewpoint       2190 non-null   float64
 7   humidity       2190 non-null   float64
 8   cloud          2190 non-null   float64
 9   sunshine       2190 non-null   float64
 10  winddirection  2190 non-null   float64
 11  windspeed      2190 non-null   float64
 12  rainfall       2190 non-null   int64  
dtypes: float64(10), int64(3)
memory usage: 222.5 KB


In [9]:
train_std=df_train.drop(['id','rainfall'],axis=1).describe().iloc[2,]
train_std

day              105.203592
pressure           5.655366
maxtemp            5.654330
temparature        5.222410
mintemp            5.059120
dewpoint           5.288406
humidity           7.800654
cloud             18.026498
sunshine           3.626327
winddirection     80.002416
windspeed          9.898659
Name: std, dtype: float64

In [10]:
# origin_train=pd.concat([origin_std,train_std],axis=1)
# origin_train.columns=['origin_std','train_std']

In [11]:
# origin_train['diff']=origin_train['origin_std']-origin_train['train_std']
# origin_train

In [12]:
df_test.describe()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed
count,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,730.000000,729.000000,730.000000
mean,2554.500000,183.000000,1013.503014,26.372466,23.963288,22.110274,20.460137,82.669863,76.360274,3.664384,103.923182,22.484247
std,210.877136,105.438271,5.505871,5.672521,5.278098,5.170744,5.391169,7.818714,17.934121,3.639272,81.695458,9.954779
min,2190.000000,1.000000,1000.000000,7.400000,5.900000,4.200000,-0.000000,39.000000,0.000000,0.000000,10.000000,4.500000
25%,2372.250000,92.000000,1008.725000,21.600000,19.825000,17.825000,16.800000,77.250000,69.000000,0.325000,40.000000,14.500000
50%,2554.500000,183.000000,1012.700000,27.800000,25.650000,23.900000,22.300000,82.000000,83.000000,2.200000,70.000000,21.300000
75%,2736.750000,274.000000,1017.600000,31.000000,28.375000,26.400000,25.000000,89.000000,88.000000,6.675000,200.000000,28.400000
max,2919.000000,365.000000,1032.200000,35.800000,31.800000,29.100000,26.700000,98.000000,100.000000,11.800000,300.000000,59.500000


# Feature Engineering

**Statistics**

In [13]:
#Add moving average 4
df_train['temp_mean_4'] = df_train['temparature'].rolling(window=4).mean().bfill().ffill()
df_train['windirection_mean_4'] = df_train['winddirection'].rolling(window=4).mean().bfill().ffill()
df_train['humidity_mean_4'] = df_train['humidity'].rolling(window=4).mean().bfill().ffill()
df_train['cloud_mean_4']=df_train['cloud'].rolling(window=4).mean().bfill().ffill()
df_train['windspeed_mean_4']=df_train['windspeed'].rolling(window=4).mean().bfill().ffill()


In [14]:
df_train['range_temp']=df_train['maxtemp']-df_train['mintemp']
df_train['range_temp_mean']=df_train['range_temp'].rolling(window=4).mean().bfill().ffill()

#do timeseries shift 
df_train['pressure_shift']=df_train['pressure'].shift(1).bfill()
df_train['humidity_shift']=df_train['humidity'].shift(1).bfill()
df_train['cloud_shift']=df_train['cloud'].shift(1).bfill()
df_train['pressure_shift2']=df_train['pressure'].shift(2).bfill()
df_train['humidity_shift2']=df_train['humidity'].shift(2).bfill()
df_train['cloud_shift2']=df_train['cloud'].shift(2).bfill()



In [15]:
df_test['temp_mean_4'] = df_test['temparature'].rolling(window=4).mean().bfill().ffill()
df_test['windirection_mean_4'] = df_test['winddirection'].rolling(window=4).mean().bfill().ffill()
df_test['humidity_mean_4'] = df_test['humidity'].rolling(window=4).mean().bfill().ffill()
df_test['cloud_mean_4']=df_test['cloud'].rolling(window=4).mean().bfill().ffill()
df_test['windspeed_mean_4']=df_test['windspeed'].rolling(window=4).mean().bfill().ffill()


In [16]:
df_test['range_temp']=df_test['maxtemp']-df_test['mintemp']
df_test['range_temp_mean']=df_test['range_temp'].rolling(window=4).mean().bfill().ffill()

#do timeseries shift 
df_test['pressure_shift']=df_test['pressure'].shift(1).bfill()
df_test['humidity_shift']=df_test['humidity'].shift(1).bfill()
df_test['cloud_shift']=df_test['cloud'].shift(1).bfill()
df_test['pressure_shift2']=df_test['pressure'].shift(2).bfill()
df_test['humidity_shift2']=df_test['humidity'].shift(2).bfill()
df_test['cloud_shift2']=df_test['cloud'].shift(2).bfill()



**Sinc and Cosine transformation for cyclic**

In [17]:
df_train['monthly_sin'] = np.sin(2 * np.pi * df_train['day'] / 30.44)
df_train['monthly_cos'] = np.cos(2 * np.pi * df_train['day'] / 30.44)
df_train['quarterly_sin'] = np.sin(2 * np.pi * df_train['day'] / 91.31)
df_train['quarterly_cos'] = np.cos(2 * np.pi * df_train['day'] / 91.31)
df_train['yearly_sin'] = np.sin(2 * np.pi * df_train['day'] / 365.25)
df_train['yearly_cos'] = np.cos(2 * np.pi * df_train['day'] / 365.25)



In [18]:
df_test['monthly_sin'] = np.sin(2 * np.pi * df_test['day'] / 30.44)
df_test['monthly_cos'] = np.cos(2 * np.pi * df_test['day'] / 30.44)
df_test['quarterly_sin'] = np.sin(2 * np.pi * df_test['day'] / 91.31)
df_test['quarterly_cos'] = np.cos(2 * np.pi * df_test['day'] / 91.31)
df_test['yearly_sin'] = np.sin(2 * np.pi * df_test['day'] / 365.25)
df_test['yearly_cos'] = np.cos(2 * np.pi * df_test['day'] / 365.25)


In [19]:
#Select columns to be dropped
#drop_cols=['id','temparature','winddirection','windspeed','range_temp']
drop_cols=['id']
df_train=df_train.drop(drop_cols,axis=1)
df_test=df_test.drop(drop_cols,axis=1)

In [20]:
df_test[df_test['winddirection'].isna()]

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,...,cloud_shift,pressure_shift2,humidity_shift2,cloud_shift2,monthly_sin,monthly_cos,quarterly_sin,quarterly_cos,yearly_sin,yearly_cos
517,153,1007.8,32.9,30.6,28.9,22.0,65.0,75.0,8.2,NaN,...,87.0,1008.4,72.0,22.0,0.16438,0.986397,-0.892742,-0.450569,0.487847,-0.872929


In [21]:
df_test['winddirection']=df_test['winddirection'].fillna(100)

In [22]:
sc = StandardScaler()
Y_train = df_train['rainfall']
X_train=sc.fit_transform(df_train.drop('rainfall',axis=1))
X_test=sc.fit_transform(df_test)
# X_val = X_train[-730:,]
# Y_val = Y_train[-730:,]
# X_train=X_train[:-730,]
# Y_train=Y_train[:-730,]



In [23]:
from sklearn.model_selection import TimeSeriesSplit

class Rainfall_TS_Dataset(Dataset):
    def __init__(self, X, y, sequence_length, n_splits=5,device='cpu'):
        """
        Args:
            X (np.array): Feature data of shape (num_samples, num_features).
            y (np.array): Label data of shape (num_samples,).
            sequence_length (int): Length of each sequence.
            n_splits (int): Number of splits for TimeSeriesSplit.
        """
        self.X = X
        self.y = y
        self.sequence_length = sequence_length
        self.n_splits = n_splits
        
        # Create sequences
        self.X_sequences = np.array([X[i:i+sequence_length] for i in range(len(X) - sequence_length)])
        self.y_sequences = np.array([y[i+sequence_length-1] for i in range(len(y) - sequence_length)])
        
        # Initialize TimeSeriesSplit
        self.tscv = TimeSeriesSplit(n_splits=n_splits)
        self.splits = list(self.tscv.split(self.X_sequences))
        self.current_fold = 0
        self.device=device
    
    def set_fold(self, fold):
        """Set the current fold for training/validation."""
        if fold >= self.n_splits:
            raise ValueError(f"Fold {fold} is out of range. Maximum folds: {self.n_splits}")
        self.current_fold = fold
    
    def __len__(self):
        """Return the number of samples in the current fold."""
        train_index, val_index = self.splits[self.current_fold]
        return len(train_index)  # Return size of training set
    
    def __getitem__(self, idx):
        """Return a batch of sequences and labels for the current fold."""
        train_index, val_index = self.splits[self.current_fold]
        X_train = self.X_sequences[train_index]
        y_train = self.y_sequences[train_index]
        
        # Convert to PyTorch tensors
        X_train_tensor = torch.tensor(X_train[idx], dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train[idx], dtype=torch.float32)
        # Add extra dimension for label if necessary
        if y_train_tensor.dim() == 0:  # If scalar
            y_train_tensor = y_train_tensor.unsqueeze(0)  # Add dimension at index 0
        else:
            y_train_tensor = y_train_tensor.unsqueeze(1)  # Add dimension at index 1
        
        
        return X_train_tensor.to(self.device), y_train_tensor.to(self.device)

    def get_validation_data(self):
        """Return the validation set for the current fold."""
        _, val_index = self.splits[self.current_fold]
        X_val = self.X_sequences[val_index]
        y_val = self.y_sequences[val_index]
        
        # Convert to PyTorch tensors
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
        # Add extra dimension for label if necessary
        if y_val_tensor.dim() == 0:  # If scalar
            y_val_tensor = y_val_tensor.unsqueeze(0)  # Add dimension at index 0
        else:
            y_val_tensor = y_val_tensor.unsqueeze(1)  # Add dimension at index 1
        
        
        return X_val_tensor.to(self.device), y_val_tensor.to(self.device)

In [24]:
class RainfallTest(Dataset):

    def __init__(self,X_test,device='cpu'):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_test.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_test).float() # size [n_samples, n_features]
        self.labels = df_test.columns
        self.device = device
        
    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index].to(self.device)

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [25]:
class RainfallValidate(Dataset):

    def __init__(self):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_val.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_val).float() # size [n_samples, n_features]
        self.labels = df_train.drop('rainfall',axis=1).columns
        self.y_data = torch.from_numpy(Y_val.values).float() # size [n_samples, 1]
        self.y_label = 'rainfall'

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [26]:
class RainfallDataset(Dataset):

    def __init__(self):
        # Initialize data, download, etc.
        # read with numpy or pandas
        #xy = np.loadtxt('./data/wine/wine.csv', delimiter=',', dtype=np.float32, skiprows=1)
        self.n_samples = X_train.shape[0]

        # here the first column is the class label, the rest are the features
        self.x_data = torch.from_numpy(X_train).float() # size [n_samples, n_features]
        self.labels = df_train.drop('rainfall',axis=1).columns
        self.y_data = torch.from_numpy(Y_train.values).float() # size [n_samples, 1]
        self.y_label = 'rainfall'

    # support indexing such that dataset[i] can be used to get i-th sample
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def getlabels(self, index):
        return self.labels[index]
    # we can call len(dataset) to return the size
    def __len__(self):
        return self.n_samples

In [27]:
#Use dataloader
batch_size=1
n_splits=2
train_dataset = Rainfall_TS_Dataset(X_train,Y_train,730,n_splits)
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=False,
                          num_workers=1)

# convert to an iterator and look at one random sample


In [28]:
import torch.optim as optim

class Rainfall_LSTM(nn.Module):
    def __init__(self, input_dim=22, hidden_dim=128, output_dim=1, num_layers=2, dropout=0.3,bidirectional=False):
        super(Rainfall_LSTM, self).__init__()
        
        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_dim,  # 22 input features
            hidden_size=hidden_dim,  # Hidden state size nn.Linear(hidden_size * 2 if bidirectional else hidden_size, 1)
            num_layers=num_layers,   # Number of LSTM layers
            batch_first=True,        # Input shape: (batch, seq_len, input_dim)
            dropout=dropout,         # Dropout for regularization
            bidirectional=bidirectional      # Bi-direction LSTM
        )
        
        # Batch normalization after LSTM
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2 if bidirectional else hidden_dim)
        
        # Fully connected layer to map LSTM output to 1 output
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, 1) #nn.Linear(hidden_dim, output_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Input x has shape (batch_size, 22)
        # Add a sequence dimension to make it (batch_size, 1, 22)
        #x = x.unsqueeze(1)  # Shape: (batch_size, 1, 22)
        
        # LSTM layer
        lstm_out, _ = self.lstm(x)  # lstm_out shape: (batch_size, 1, hidden_dim)
        
        # Take the output of the last time step
        lstm_out = lstm_out[:, -1, :]  # Shape: (batch_size, hidden_dim)
        
        # Batch normalization (skip if batch size is 1 during training)
        if lstm_out.size(0) > 1 or not self.training:
            lstm_out = self.bn1(lstm_out)
        
        # Dropout
        lstm_out = self.dropout(lstm_out)
        
        # Fully connected layer
        output = self.fc(lstm_out)  # Shape: (batch_size, 1)
        
        return output  # Return raw output 

In [29]:
if device.type=='cuda':
    # Check available GPUs
    num_gpus = torch.cuda.device_count()
    print(f"Number of available GPUs: {num_gpus}")
else:
    num_gpus=0

Number of available GPUs: 2


In [30]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import CosineAnnealingLR

def objective(trial):
    try:
        print(f"Running Trial {trial.number}")
        print(f"Pruner: {study.pruner}")  # Should print `NopPruner()`
        input_dim = X_train.shape[1]
        hidden_dim = 180
        output_dim = 1
        num_layers = 2
        dropout = trial.suggest_float("dropout",0.20,0.30,log=True)
        bidirectional = False
        #batch_size = 365
        if device.type=='cuda':
            learning_rate=trial.suggest_float("learning_rate",0.00001,0.0001,log=True)
        else:
            learning_rate=trial.suggest_float("learning_rate",0.001,0.01,log=True)
        num_epochs = 20  # Each Optuna trial is 20 sets 
        patience=5
        if device.type=='cuda':
            weight_decay = trial.suggest_float("weight_decay",1e-5,1e-4,log=True) #6e-5 # L2 regularization strength
        else:
            weight_decay = trial.suggest_float("weight_decay",1e-4,1e-3,log=True) #1e-3 # L2 regularization strength
        
        
        # Initialize model, loss, and optimizer
        model = Rainfall_LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers, dropout=dropout,bidirectional=bidirectional)
        if num_gpus>1:
             model = nn.DataParallel(model)
        
        model=model.to(device)
        criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss
    
        optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
        optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=learning_rate,weight_decay=weight_decay)
        #optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        
        # Initialize learning rate scheduler
        sch_factor=trial.suggest_float("sch_factor",0.1,0.2,log=False)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=patience)
        
        # Initialize CosineAnnealingLR scheduler
        #scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)
        # Create DataLoader
        dataloader = train_loader  
    
        # Training loop
        total_epoch_loss=0.0
        for epoch in range(num_epochs):
           # model.train()  # Set model to training mode        
            total_val_loss = 0.0
            for fold in range(n_splits):
                #print(f"Training on Fold {fold + 1}")
                train_dataset.set_fold(fold)  # Set the current fold
                model.train()
                for batch_idx, batch in enumerate(dataloader):
                    inputs, targets = batch  # features is a list of 23 tensors, targets is a list of 1 tensors
                    inputs = inputs.to(device)
                    targets = targets.to(device)
            
                    outputs = model(inputs)  # Shape: (batch_size, output_dim)
                    loss = criterion(outputs, targets.reshape(-1,1))  # Compute loss                
                    # Backward pass and optimization
                    optimizer.zero_grad()  # Clear gradients
                    loss.backward()  # Backpropagation
                    optimizer.step()  # Update weights
            
                # Validation phase
                model.eval()  # Set model to evaluation mode
                val_loss = 0.0
                with torch.no_grad():  # Disable gradient computation
                    X_val_tensor, y_val_tensor = train_dataset.get_validation_data()  # Get validation set
                    X_val_tensor=X_val_tensor.to(device)
                    y_val_tensor=y_val_tensor.to(device)            
                    outputs = model(X_val_tensor)  # Shape: (batch_size, 1)                    
                    # Compute loss
                    loss = criterion(outputs, y_val_tensor)  # Regression or binary classification loss
                    total_val_loss += loss.item()
                    #print(f"Epoch [{epoch+1}], Fold [{fold+1}/{n_splits}], Validation Loss: {val_loss:.4f}")            
            # Compute average validation loss
            avg_val_loss = total_val_loss / n_splits
            if (epoch+1)%10==0:
               print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}")        
            # Update learning rate scheduler
            scheduler.step(avg_val_loss)  # Pass validation loss to scheduler
            #scheduler.step()
            trial.report(avg_val_loss,epoch+1)
            total_epoch_loss+=avg_val_loss
    
            # Handle pruning based on the intermediate value.
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        print(f"average loss per {num_epochs} epoch is {total_epoch_loss/num_epochs}")    
    except Exception as e:
        print(f"⛔ Trial {trial.number} crashed: {str(e)}")
        raise  # Re-raise to see traceback      
    return total_epoch_loss/num_epochs
 

In [31]:
def OptunaTrial(n_trials):
    study = optuna.create_study(direction="minimize",pruner=optuna.pruners.NopPruner())
    study.optimize(objective, n_trials=n_trials,timeout=None,catch=())

    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("  Number of pruned trials: ", len(pruned_trials))
    print("  Number of complete trials: ", len(complete_trials))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))    

In [32]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import CosineAnnealingLR

def CreateModel(params):
    input_dim = X_train.shape[1]
    hidden_dim = 180
    output_dim = 1
    num_layers = 2
    dropout = params['dropout']
    bidirectional = False
    #batch_size = 365
    if device.type=='cuda':
        learning_rate=params['lr']
    else:
        learning_rate = 0.006
    #num_epochs = 100
    patience=5
    if device.type=='cuda':
        weight_decay = params['weight']#6e-5 # L2 regularization strength
    else:
        weight_decay = 1e-3 # L2 regularization strength
    
    
    # Initialize model, loss, and optimizer
    model = Rainfall_LSTM(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_layers=num_layers, dropout=dropout,bidirectional=bidirectional)
    if num_gpus>1:
         model = nn.DataParallel(model)
    
    model=model.to(device)
    criterion = nn.BCEWithLogitsLoss()  # Binary Cross-Entropy Loss
    optimizer = getattr(optim,params['optim'])(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # Initialize learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=params['sch_factor'], patience=patience)
    return {'model':model,'loss':criterion,'optim':optimizer,'schedule':scheduler}
    

In [33]:
# Count layers recursively (excluding the container)
# recursive_layers = sum(1 for _ in model.children())
# print(f"Number of recursive layers: {recursive_layers}")

# # Count all submodules
# all_layers = sum(1 for _ in model.modules())
# print(f"Number of all submodules: {all_layers}")

# # Count learnable layers
# learnable_layers = sum(1 for _ in model.parameters())
# print(f"Number of learnable layers: {learnable_layers}")

# for name, param in model.named_parameters():
#     print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

In [34]:
def Training(model,criterion,optimizer,scheduler,num_epochs=1):
    # Initialize variables to track the best model
    best_loss = float('inf')  # For regression (lower is better)
    # best_accuracy = 0.0  # For classification (higher is better)
    best_model_state = None
    
    # Training loop
    for epoch in range(num_epochs):
       # model.train()  # Set model to training mode
        
        total_val_loss = 0.0
        for fold in range(n_splits):
            #print(f"Training on Fold {fold + 1}")
            train_dataset.set_fold(fold)  # Set the current fold
            model.train()
            for batch_idx, batch in enumerate(train_loader):
                inputs, targets = batch  # features is a list of 23 tensors, targets is a list of 1 tensors
                
                # Stack features and targets into batches
                #features = torch.stack(features)  # Shape: (23, 22)
                #targets = torch.stack(targets)    # Shape: (23, 1)
                inputs = inputs.to(device)
                targets = targets.to(device)
                #inputs = inputs.float()  # Convert inputs to float32
                #targets = targets.float()  # Convert targets to float32
        
                outputs = model(inputs)  # Shape: (batch_size, output_dim)
                loss = criterion(outputs, targets.reshape(-1,1))  # Compute loss
                
                # Backward pass and optimization
                optimizer.zero_grad()  # Clear gradients
                loss.backward()  # Backpropagation
                optimizer.step()  # Update weights
        
            # Validation phase
            model.eval()  # Set model to evaluation mode
            val_loss = 0.0
            with torch.no_grad():  # Disable gradient computation
                X_val_tensor, y_val_tensor = train_dataset.get_validation_data()  # Get validation set
                X_val_tensor=X_val_tensor.to(device)
                y_val_tensor=y_val_tensor.to(device)
                
                outputs = model(X_val_tensor)  # Shape: (batch_size, 1)
                    
                # Compute loss
                loss = criterion(outputs, y_val_tensor)  # Regression or binary classification loss
                total_val_loss += loss.item()
                #print(f"Epoch [{epoch+1}], Fold [{fold+1}/{n_splits}], Validation Loss: {val_loss:.4f}")
            
        # Compute average validation loss
        avg_val_loss = total_val_loss / n_splits
        if (epoch+1)%20==0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_val_loss:.4f}")
        
        # Update learning rate scheduler
        scheduler.step(avg_val_loss)  # Pass validation loss to scheduler
        #scheduler.step()
        
        # Save the best model
        if avg_val_loss < best_loss:  # For regression (lower is better)
            print(f"Validation loss improved from {best_loss:.4f} to {avg_val_loss:.4f}. Saving model...")
            best_loss = avg_val_loss
            best_model_state = model.state_dict()  # Save the model's state dictionary
            torch.save(best_model_state, "best_model.pth")  # Save to file
    
    # Load the best model
    model.load_state_dict(torch.load("best_model.pth"))
    print(f"Best model loaded. with best loss {best_loss}")
    print("Training complete!")
    return model

In [35]:
# Best params from Optuna
# dropout: 0.25407049744180404
 #    learning_rate: 2.9040091641250882e-05
 #    weight_decay: 1.7261419417511937e-05
 #    optimizer: RMSprop
 #    sch_factor: 0.18167068626401306
params={'dropout':0.25407049744180404,
        'lr':2.9040091641250882e-05,
        'weight':1.7261419417511937e-05,
        'optim':'RMSprop',
        'sch_factor':0.18167068626401306}
model_dict=CreateModel(params)

model=Training(model_dict['model'],model_dict['loss'],model_dict['optim'],model_dict['schedule'],50)
#OptunaTrial(3)


Validation loss improved from inf to 0.5528. Saving model...
Validation loss improved from 0.5528 to 0.5169. Saving model...
Validation loss improved from 0.5169 to 0.5067. Saving model...
Validation loss improved from 0.5067 to 0.4991. Saving model...
Validation loss improved from 0.4991 to 0.4915. Saving model...
Validation loss improved from 0.4915 to 0.4816. Saving model...
Validation loss improved from 0.4816 to 0.4677. Saving model...
Validation loss improved from 0.4677 to 0.4439. Saving model...
Validation loss improved from 0.4439 to 0.4118. Saving model...
Validation loss improved from 0.4118 to 0.3919. Saving model...
Validation loss improved from 0.3919 to 0.3883. Saving model...
Epoch [20/50], Validation Loss: 0.4765
Epoch [40/50], Validation Loss: 0.4972
Best model loaded. with best loss 0.3882573992013931
Training complete!


<ipython-input-34-e1ef247bdb1e>:67: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model.pth"))


In [36]:
#model=Training(model_dict['model'],model_dict['loss'],model_dict['optim'],model_dict['schedule'],1)

In [37]:
test_dataset=RainfallTest(X_test,device)
test_dataloader=DataLoader(test_dataset,batch_size=730,shuffle=False)
features = next(iter(test_dataloader))
#model.predict(features)

In [38]:
model.eval()

features=features.unsqueeze(1).to(device)
with torch.no_grad():
    logits = model(features)  # Shape: (1, 1)

# Convert logits to probabilities using sigmoid
probabilities = torch.sigmoid(logits)  # Shape: (1, 1)

print("Is there any null value in Output?")
print(np.isnan(probabilities.cpu()).any())
#threshold = 0.5
#binary_output1 = (probabilities>threshold).float()

#print("Logits:", logits)
#print("Probabilities:", probabilities)

Is there any null value in Output?
tensor(0, dtype=torch.uint8)


In [39]:
torch.isnan(probabilities).any()

tensor(False, device='cuda:0')

In [40]:
temp=pd.concat([df_sub.drop('rainfall',axis=1),pd.DataFrame(probabilities.cpu(),columns=['rainfall'])],axis=1)
temp.to_csv("submission.csv",index=False)

In [41]:
temp.isna().any()

id          False
rainfall    False
dtype: bool

In [42]:
# features = next(iter(test_dataloader))
# with torch.no_grad():
#     logits = model(features.unsqueeze(1).to(device))  # Shape: (1, 1)

# # Convert logits to probabilities using sigmoid
# probabilities = torch.sigmoid(logits)  # Shape: (1, 1)
# threshold = 0.5
# binary_output2 = (probabilities>threshold).float()


In [43]:
# if binary_output1.device.type=='cpu':
#     temp=np.concatenate((binary_output1,binary_output2))
# else:
#     temp=np.concatenate((binary_output1.cpu(),binary_output2.cpu()))
# temp.shape

In [44]:
# temp2 = pd.concat([df_sub.drop('rainfall',axis=1),pd.DataFrame(temp,columns=['rainfall'])],axis=1)

In [45]:
# temp2.to_csv("submission_ts_x_gpu.csv",index=False)